# Risk Contributions and Euler Allocation

Portfolio volatility, VaR and expected shortfall (05/01, 05/02) are single numbers. A risk
committee asks the next question: **which positions** produce that number? Capital weights do
not answer it: a 60/40 equity/bond portfolio gets about 94% of its risk from equities. Euler
allocation splits a risk measure into contributions that add up exactly, using nothing more
than partial derivatives.

## 0. Math link

- [Partial derivatives](https://github.com/dnlwrthstr/LearningCalculus/blob/main/coursera/02_gradients/01_partial_derivatives_part_1.ipynb)
  and [gradients](https://github.com/dnlwrthstr/LearningCalculus/blob/main/coursera/02_gradients/03_gradients.ipynb):
  the marginal risk of a position is $\partial\rho/\partial w_i$, and the vector of them is the gradient $\nabla\rho(w)$.
- [Chain rule](https://github.com/dnlwrthstr/LearningCalculus/blob/main/coursera/00_derivatives/16_properties_of_the_derivative_chain_rule.ipynb):
  the proof of Euler's theorem differentiates $\rho(\lambda w)$ with respect to $\lambda$.
- Portfolio variance $w^\top\Sigma w$ from [04/01](../04_portfolio_mathematics/01_portfolio_variance.ipynb),
  VaR and ES from [05/01](./01_value_at_risk.ipynb) and [05/02](./02_expected_shortfall.ipynb).

## 1. Introduction

"Bonds are 30% of the portfolio" is a statement about **capital**. "Bonds are 3% of the risk" is a
statement about **risk**, and it is the one that matters when deciding where a loss would come from,
which positions to cut, or how to set risk budgets for managers.

After this notebook you can:

- state and prove Euler's theorem for positively homogeneous risk measures;
- compute marginal and total risk contributions for volatility, normal VaR, and simulated ES;
- read a negative contribution as a hedge;
- check optimality of a portfolio with the ratio of excess return to marginal risk;
- explain why risk contributions are not the same as stand-alone risks.

## 2. Theory

### Homogeneous risk measures

A risk measure $\rho(w)$ of a portfolio with position vector $w$ is **positively homogeneous of
degree 1** if doubling every position doubles the risk:

$$
\rho(\lambda w) = \lambda\,\rho(w) \qquad \text{for } \lambda > 0
$$

Volatility $\sigma(w) = \sqrt{w^\top\Sigma w}$, normal VaR $z_\alpha\sigma(w)\,V$, and ES all have this property
(05/02, coherence axiom 3). Variance does **not**: it scales with $\lambda^2$.

### Euler's theorem

If $\rho$ is positively homogeneous of degree 1 and differentiable, then

$$
\rho(w) = \sum_i w_i\,\frac{\partial\rho}{\partial w_i}(w)
$$

**Proof.** Differentiate both sides of $\rho(\lambda w) = \lambda\rho(w)$ with respect to $\lambda$. By the chain rule
the left side gives $\sum_i w_i\,\frac{\partial\rho}{\partial w_i}(\lambda w)$, the right side $\rho(w)$. Set $\lambda = 1$. $\square$

This motivates the definitions:

- **marginal risk** $\mathrm{MR}_i = \partial\rho/\partial w_i$: the change in portfolio risk per unit of extra
  position $i$;
- **risk contribution** $\mathrm{RC}_i = w_i\,\mathrm{MR}_i$, with $\sum_i \mathrm{RC}_i = \rho(w)$;
- **percentage contribution** $\mathrm{RC}_i / \rho(w)$, summing to 100%.

### Volatility contributions

With $\sigma(w) = (w^\top\Sigma w)^{1/2}$, the chain rule gives

$$
\nabla\sigma(w) = \frac{\Sigma w}{\sigma(w)}, \qquad
\mathrm{RC}_i = \frac{w_i\,(\Sigma w)_i}{\sigma(w)}
$$

and $\sum_i \mathrm{RC}_i = w^\top\Sigma w/\sigma = \sigma$. Two equivalent readings:

- $(\Sigma w)_i = \operatorname{Cov}(R_i, R_p)$, so $\mathrm{RC}_i = w_i\,\beta_{i,p}\,\sigma_p$ with $\beta_{i,p} = \operatorname{Cov}(R_i, R_p)/\sigma_p^2$, the beta of asset $i$
  on **the portfolio** (not on the market).
- $\mathrm{RC}_i = w_i\,\sigma_i\,\rho_{i,p}$: the stand-alone risk $w_i\sigma_i$ scaled by the correlation of asset $i$ with the
  portfolio. Since $\rho_{i,p} \le 1$, a contribution never exceeds the stand-alone risk; the gap is the
  diversification. If $\rho_{i,p} < 0$ the contribution is **negative**: the position is a hedge.

### VaR and ES contributions

Normal VaR is $z_\alpha V\sigma(w)$, so its contributions are the volatility contributions times $z_\alpha V$; the
percentages are identical. For a general loss $L = -\sum_i w_i R_i$ with continuous distribution,

$$
\frac{\partial\operatorname{ES}_\alpha}{\partial w_i} = E\big[-R_i \mid L \ge \operatorname{VaR}_\alpha\big], \qquad
\mathrm{RC}_i^{\mathrm{ES}} = E\big[-w_i R_i \mid L \ge \operatorname{VaR}_\alpha\big]
$$

(stated here; Tasche 1999). The ES contribution of a position is its **average loss on the
portfolio's worst days**. With simulated scenarios: average $-w_i R_i$ over the scenarios in the
portfolio's tail. These contributions add up to the ES exactly, and they see what volatility misses:
positions that lose precisely when the portfolio does.

### Marginal risk and optimality

Move a small amount $\delta$ from asset $j$ into asset $i$. Expected return changes by $\delta(\mu_i - \mu_j)$, risk by
about $\delta(\mathrm{MR}_i - \mathrm{MR}_j)$. If $(\mu_i - r_f)/\mathrm{MR}_i > (\mu_j - r_f)/\mathrm{MR}_j$, the move improves return per
unit of risk. At the tangency portfolio (04/04) no such move helps, so

$$
\frac{\mu_i - r_f}{\mathrm{MR}_i} = \text{Sharpe ratio of the portfolio} \qquad \text{for every asset } i
$$

Dividing excess return by marginal risk asset by asset tells you which way to tilt a portfolio
that is **not** optimal. This is the basis of risk budgeting.

### What contributions are not

- They are **marginal**, not "what would happen if we sold the position": selling all of asset
  $i$ changes risk by a different amount (Python section).
- They depend on the whole portfolio: the same bond position contributes differently in a
  portfolio with more or fewer equities.
- Equal capital is not equal risk. Equal **risk** contributions is risk parity (module 08/03).

## 3. Worked Examples

### Example 1: 60/40

Swiss equities $\sigma_E = 16\%$, Swiss bonds $\sigma_B = 5\%$, correlation 0.1, weights 60/40.

$$
\Sigma w = \begin{pmatrix} 0.0256 & 0.0008 \\ 0.0008 & 0.0025 \end{pmatrix}\begin{pmatrix} 0.6 \\ 0.4 \end{pmatrix}
= \begin{pmatrix} 0.01568 \\ 0.00148 \end{pmatrix}, \qquad
\sigma^2 = 0.6 \cdot 0.01568 + 0.4 \cdot 0.00148 = 0.0100
$$

so $\sigma = 10.0\%$ a year.

| | weight | MR $= (\Sigma w)_i/\sigma$ | RC $= w_i\,\mathrm{MR}_i$ | share of risk |
|---|---|---|---|---|
| equities | 60% | 0.1568 | 9.41% | **94.1%** |
| bonds | 40% | 0.0148 | 0.59% | **5.9%** |
| total | 100% | | 10.00% | 100% |

A "balanced" 60/40 portfolio is an equity portfolio in risk terms.

### Example 2: the 40/30/30 portfolio of module 04

Volatilities 16%, 17%, 5%, correlations 0.8 (Swiss–global equities), 0.1 (Swiss equities–bonds), 0.

$$
\Sigma w = (0.017008,\ 0.017374,\ 0.001070), \qquad \sigma^2 = 0.012336, \quad \sigma = 11.11\%
$$

| | weight | stand-alone $w_i\sigma_i$ | RC | share |
|---|---|---|---|---|
| Swiss equities | 40% | 6.40% | 6.12% | 55.1% |
| Global equities | 30% | 5.10% | 4.69% | 42.3% |
| Swiss bonds | 30% | 1.50% | 0.29% | 2.6% |
| total | | 13.00% | 11.11% | 100% |

The stand-alone risks add up to 13.0%; the contributions add up to the actual 11.1%. Bonds lose most
of their stand-alone risk (1.50% → 0.29%) because they are almost uncorrelated with an
equity-dominated portfolio.

### Example 3: using marginal risk

In Example 2, move 1 percentage point from bonds to Swiss equities. Marginal risks
$\mathrm{MR}_{SE} = 0.017008/0.1111 = 0.1531$, $\mathrm{MR}_B = 0.00107/0.1111 = 0.0096$, so

$$
\Delta\sigma \approx 0.01\,(0.1531 - 0.0096) = 0.14 \text{ percentage points}
$$

from 11.11% to about 11.25% (the Python section checks the exact value).

### Example 4: Euler for a simple function

$f(x, y) = \sqrt{x^2 + y^2}$ is homogeneous of degree 1. $\partial f/\partial x = x/f$, $\partial f/\partial y = y/f$, and
$x\,\partial f/\partial x + y\,\partial f/\partial y = (x^2 + y^2)/f = f$. Volatility of two **uncorrelated** assets with unit volatility
is exactly this function.

## 4. Finance Application

### Where risk contributions are used

- **Risk reports**: every institutional risk system reports contributions to volatility,
  tracking error, VaR or ES by asset class, sector, factor and manager. They add up to the total,
  which is why Euler allocation is the standard.
- **Risk budgeting**: a pension fund allocates, say, 60% of its risk budget to equities and 20% to
  credit, and checks the contributions against it, rather than the capital weights.
- **Bank capital allocation**: economic capital (ES or VaR based) is allocated to business units
  with Euler contributions; return on risk-adjusted capital (RORAC) per unit then tells where to grow.
  Tasche showed Euler is the only allocation for which "grow units with above-average RORAC"
  always improves the total.
- **Risk parity and equal-risk-contribution portfolios** (module 08/03) set the contributions equal.

### The usual misreadings

- "Bonds are 30% of the portfolio, so 30% of the risk." Example 1: 40% of capital, 6% of the risk.
- "Bonds contribute 0.3% of volatility, so selling them would cut volatility by 0.3%." Contributions
  are marginal. Selling all bonds and holding cash raises the equity share and changes every other
  contribution; for a large trade, recompute.
- "A negative contribution means the position loses money." It means the position **reduces**
  portfolio risk at the margin; it says nothing about expected return.

## 5. Python Examples

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

rng = np.random.default_rng(42)
trading_days = 252

### Volatility contributions (Examples 1 and 2)

In [ ]:
def vol_contributions(w, Sigma):
    # Euler contributions to sigma(w) = sqrt(w' Sigma w); same units as sigma
    w = np.asarray(w, dtype=float)
    sigma = np.sqrt(w @ Sigma @ w)
    marginal = Sigma @ w / sigma
    return sigma, marginal, w * marginal


def contribution_table(w, Sigma, names):
    sigma, marginal, rc = vol_contributions(w, Sigma)
    vols = np.sqrt(np.diag(Sigma))
    return pd.DataFrame({
        "weight": w,
        "stand-alone w·σ": w * vols,
        "marginal risk": marginal,
        "contribution": rc,
        "share of risk": rc / sigma,
        "beta to portfolio": Sigma @ w / sigma**2,
        "corr to portfolio": Sigma @ w / (vols * sigma),
    }, index=names)


Sigma_6040 = np.array([[0.16**2, 0.1 * 0.16 * 0.05], [0.1 * 0.16 * 0.05, 0.05**2]])
t1 = contribution_table(np.array([0.6, 0.4]), Sigma_6040, ["equities", "bonds"])
print(f"60/40 volatility: {vol_contributions([0.6, 0.4], Sigma_6040)[0]:.2%}")
t1.round(4)

In [ ]:
assets = ["Swiss equities", "Global equities", "Swiss bonds"]
mu = np.array([0.065, 0.07, 0.02])
vols = np.array([0.16, 0.17, 0.05])
corr = np.array([[1.0, 0.8, 0.1], [0.8, 1.0, 0.0], [0.1, 0.0, 1.0]])
Sigma = np.diag(vols) @ corr @ np.diag(vols)
w = np.array([0.4, 0.3, 0.3])
rf = 0.005

t2 = contribution_table(w, Sigma, assets)
sigma_p = vol_contributions(w, Sigma)[0]
print(f"portfolio volatility: {sigma_p:.2%}, sum of contributions: {t2['contribution'].sum():.2%}, "
      f"sum of stand-alone: {t2['stand-alone w·σ'].sum():.2%}")
t2.round(4)

### Checks: Euler by finite differences, and the marginal approximation

In [ ]:
def portfolio_vol(w):
    return np.sqrt(w @ Sigma @ w)


h = 1e-6
grad_fd = np.array([(portfolio_vol(w + h * e) - portfolio_vol(w - h * e)) / (2 * h) for e in np.eye(3)])
print("gradient, analytic:         ", np.round(Sigma @ w / sigma_p, 6))
print("gradient, finite difference:", np.round(grad_fd, 6))
print(f"sum w_i * dsigma/dw_i = {w @ grad_fd:.6f}, sigma = {sigma_p:.6f}")

# Example 3: move 1 pp from bonds to Swiss equities, then larger moves
marginal = Sigma @ w / sigma_p
for delta in [0.01, 0.05, 0.10, 0.30]:
    w_new = w + delta * np.array([1, 0, -1])
    linear = sigma_p + delta * (marginal[0] - marginal[2])
    print(f"move {delta:4.0%}: exact {portfolio_vol(w_new):.3%}, marginal approximation {linear:.3%}")

For small moves the marginal approximation is exact to two decimals; for a full switch of the bond
allocation (30 points) it is off by about 0.2 percentage points, because the marginal risks themselves change
along the way. Contributions describe the portfolio at its current weights.

### Optimality: excess return per unit of marginal risk

For the 40/30/30 portfolio and for the tangency portfolio $w_T \propto \Sigma^{-1}(\mu - r_f)$ (04/04).

In [ ]:
def return_per_marginal_risk(w, Sigma, mu, rf):
    sigma, marginal, _ = vol_contributions(w, Sigma)
    return (mu - rf) / marginal, (w @ mu - rf) / sigma


ratio_w, sharpe_w = return_per_marginal_risk(w, Sigma, mu, rf)
w_T = np.linalg.solve(Sigma, mu - rf)
w_T /= w_T.sum()
ratio_T, sharpe_T = return_per_marginal_risk(w_T, Sigma, mu, rf)

pd.DataFrame({
    "40/30/30 weight": w, "40/30/30: (μ−rf)/MR": ratio_w,
    "tangency weight": w_T, "tangency: (μ−rf)/MR": ratio_T,
}, index=assets).round(3), f"Sharpe 40/30/30 {sharpe_w:.3f}, tangency {sharpe_T:.3f}"

In the tangency portfolio every ratio equals its Sharpe ratio (0.49). In 40/30/30 the bonds earn
far more excess return per unit of marginal risk (1.56) than the equities (0.39 and 0.42), so moving
weight from equities to bonds raises the Sharpe ratio. That is the direction in which the tangency
portfolio (9/21/71) differs.

### A hedge: negative contribution

Add a short equity-index future to the 40/30/30 portfolio: −15% of value on an index with 0.95
correlation to Swiss equities and 0.85 to global equities, volatility 16%.

In [ ]:
names_h = assets + ["Short index future"]
vols_h = np.append(vols, 0.16)
corr_h = np.eye(4)
corr_h[:3, :3] = corr
corr_h[3, :3] = corr_h[:3, 3] = [0.95, 0.85, 0.05]
Sigma_h = np.diag(vols_h) @ corr_h @ np.diag(vols_h)
w_h = np.array([0.4, 0.3, 0.3, -0.15])
th = contribution_table(w_h, Sigma_h, names_h)
print(f"volatility with hedge: {vol_contributions(w_h, Sigma_h)[0]:.2%} (without: {sigma_p:.2%})")
th.round(4)

### Contributions to VaR and ES with fat tails and jump risk

Take the three assets from the multivariate $t(4)$ of 05/01 and add a put-writing strategy, now
more realistic than in 05/02: it loses 6% on the worst 0.5% of Swiss-equity days (an equity crash
is exactly when written puts pay out) and otherwise earns a premium with 0.3% daily noise.
Portfolio: 35/25/30/10, CHF 10,000,000. Compare shares of volatility, of 99% VaR (normal, so equal
to the volatility shares) and of simulated 99% ES.

In [ ]:
def multivariate_t(n, Sigma, df, rng):
    # same generator as 05/01
    L = np.linalg.cholesky(Sigma * (df - 2) / df)
    z = rng.standard_normal((n, Sigma.shape[0])) @ L.T
    chi2 = rng.chisquare(df, size=(n, 1))
    return z / np.sqrt(chi2 / df)


value = 10_000_000
n_scen = 500_000
p_crash, crash = 0.005, -0.06
R3 = mu / trading_days + multivariate_t(n_scen, Sigma / trading_days, 4, rng)
crash_day = R3[:, 0] <= np.quantile(R3[:, 0], p_crash)       # worst 0.5% of Swiss-equity days
put_r = (0.04 / trading_days - p_crash * crash
         + 0.003 * rng.standard_normal(n_scen)
         + crash * crash_day)
R = np.column_stack([R3, put_r])
names_4 = assets + ["Put-writing"]
w4 = np.array([0.35, 0.25, 0.30, 0.10])


def es_contributions(w, R, alpha, value):
    # Euler ES contributions: average position loss over the portfolio's worst (1 - alpha) scenarios
    pos_loss = -value * R * w
    port_loss = pos_loss.sum(axis=1)
    k = int(round((1 - alpha) * len(port_loss)))
    tail = np.argsort(port_loss)[-k:]
    return pos_loss[tail].mean(axis=0), port_loss[tail].mean()


rc_es, es_total = es_contributions(w4, R, 0.99, value)
S4 = np.cov(R, rowvar=False)
sigma4, _, rc_vol = vol_contributions(w4, S4)

es_table = pd.DataFrame({
    "weight": w4,
    "share of volatility": rc_vol / sigma4,
    "99% ES contribution (CHF)": rc_es,
    "share of 99% ES": rc_es / es_total,
}, index=names_4)
print(f"99% ES (daily): CHF {es_total:,.0f}; sum of contributions CHF {rc_es.sum():,.0f}")
es_table.style.format({"weight": "{:.0%}", "share of volatility": "{:.1%}",
                       "99% ES contribution (CHF)": "{:,.0f}", "share of 99% ES": "{:.1%}"})

The put writer is 10% of capital and about 3% of volatility, but about 12% of the 99% ES. Most
of the time it barely moves, so its covariance with the portfolio is small. Its crash days are the
equity crash days, which are the portfolio's worst days, and the ES contribution averages its loss
over exactly those days. A volatility-based risk report would understate this position by a
factor of four.

## 6. Visualizations

In [ ]:
portfolios = {
    "60/40\n(30/30/40)": (np.array([0.3, 0.3, 0.4])),
    "40/30/30": w,
    "equal weight": np.full(3, 1 / 3),
    "tangency": w_T,
}
colors = ["tab:red", "tab:orange", "tab:blue"]
fig, ax = plt.subplots(figsize=(11, 5))
x_pos = np.arange(len(portfolios))
for j, (name, wp) in enumerate(portfolios.items()):
    sigma_j, _, rc_j = vol_contributions(wp, Sigma)
    for k, (vals, offset) in enumerate([(wp, -0.2), (rc_j / sigma_j, 0.2)]):
        bottom = 0.0
        for i in range(3):
            ax.bar(j + offset, vals[i], bottom=bottom, width=0.35, color=colors[i],
                   alpha=0.45 if k == 0 else 0.9, label=assets[i] if (j == 0 and k == 1) else None)
            bottom += vals[i]
    ax.text(j - 0.2, 1.02, "capital", ha="center", fontsize=8)
    ax.text(j + 0.2, 1.02, "risk", ha="center", fontsize=8)
ax.set_xticks(x_pos)
ax.set_xticklabels(portfolios.keys())
ax.set_ylim(0, 1.1)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}"))
ax.set_title("Capital weights (light) vs shares of volatility (dark)")
ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1.0))
ax.grid(True, alpha=0.3, axis="y")
plt.tight_layout()
plt.show()

In [ ]:
w_eq = np.linspace(0, 1, 201)
eq_share = []
for we in w_eq:
    s, _, rc = vol_contributions(np.array([we, 1 - we]), Sigma_6040)
    eq_share.append(rc[0] / s)
eq_share = np.array(eq_share)
w_parity = 0.05 / (0.16 + 0.05)

plt.figure(figsize=(8, 5))
plt.plot(w_eq, eq_share, color="tab:red", label="equity share of risk")
plt.plot(w_eq, w_eq, color="grey", linestyle=":", label="equity share of capital")
plt.axhline(0.5, color="black", linewidth=0.8)
plt.plot(w_parity, 0.5, "o", color="black")
plt.annotate(f"equal risk at {w_parity:.0%} equities", (w_parity, 0.5), textcoords="offset points", xytext=(10, -20))
plt.plot(0.6, eq_share[120], "o", color="tab:red")
plt.annotate(f"60/40: {eq_share[120]:.0%} of risk", (0.6, eq_share[120]), textcoords="offset points", xytext=(10, -20))
plt.title("Equities (16% vol) and bonds (5% vol), correlation 0.1")
plt.xlabel("equity weight")
plt.ylabel("share")
plt.gca().xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}"))
plt.gca().yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}"))
plt.grid(True, alpha=0.3)
plt.legend(loc="lower right")
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
x_pos = np.arange(4)
axes[0].bar(x_pos - 0.27, es_table["weight"], width=0.27, label="capital")
axes[0].bar(x_pos, es_table["share of volatility"], width=0.27, label="volatility")
axes[0].bar(x_pos + 0.27, es_table["share of 99% ES"], width=0.27, label="99% ES")
axes[0].set_xticks(x_pos)
axes[0].set_xticklabels(names_4, rotation=15)
axes[0].yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}"))
axes[0].set_title("Shares of capital, volatility and ES")
axes[0].legend()
axes[0].grid(True, alpha=0.3, axis="y")

x3 = np.arange(3)
axes[1].bar(x3 - 0.2, ratio_w, width=0.4, label="40/30/30")
axes[1].bar(x3 + 0.2, ratio_T, width=0.4, label="tangency")
axes[1].axhline(sharpe_w, color="tab:blue", linestyle="--", linewidth=1, label=f"Sharpe 40/30/30 = {sharpe_w:.2f}")
axes[1].axhline(sharpe_T, color="tab:orange", linestyle="--", linewidth=1, label=f"Sharpe tangency = {sharpe_T:.2f}")
axes[1].set_xticks(x3)
axes[1].set_xticklabels(assets, rotation=15)
axes[1].set_title("Excess return per unit of marginal risk")
axes[1].legend(fontsize=8)
axes[1].grid(True, alpha=0.3, axis="y")
plt.tight_layout()
plt.show()

## 7. Exercises

### Basic

1. Compute by hand the risk contributions of a 50/50 portfolio of equities (16%) and bonds (5%)
   with correlation 0.1. Check with `contribution_table`.
2. Draw 5 random weight vectors for the three assets, including some with short positions, and
   confirm that the contributions always add up to the volatility. Can a contribution be larger than the
   stand-alone risk $|w_i|\sigma_i$?

### Intermediate

3. Interpret: a risk report says "Swiss bonds: 30% of capital, 2.6% of risk". Write two sentences a
   board member can act on, and one sentence on what the number does **not** mean.
4. For two assets, show that equal risk contributions require $w_1\sigma_1 = w_2\sigma_2$, **whatever the
   correlation**. What equity weight gives equal risk for equities (16%) and bonds (5%)?

### Advanced

5. Tracking error is $\mathrm{TE}(a) = \sqrt{a^\top\Sigma a}$ with active weights $a = w - b$. It is homogeneous of degree 1 in $a$.
   Compute the contributions to tracking error of the 40/30/30 portfolio against the benchmark
   30/40/30 (04/01). Which active position contributes most? Why is the contribution of the
   global-equity **underweight** positive, although its active weight is negative?

## 8. Solutions / Checks

In [ ]:
# 1.
contribution_table(np.array([0.5, 0.5]), Sigma_6040, ["equities", "bonds"]).round(4)

In [ ]:
# 2.
for _ in range(5):
    wr = rng.normal(0.33, 0.5, 3)
    s, _, rc = vol_contributions(wr, Sigma)
    print(f"w = {np.round(wr, 2)}, sigma {s:.4f}, sum RC {rc.sum():.4f}, "
          f"RC <= |w| sigma: {np.all(rc <= np.abs(wr) * vols + 1e-12)}")

# 4.
print(f"4. equal-risk equity weight: {0.05 / 0.21:.1%}")
t4 = contribution_table(np.array([0.05 / 0.21, 0.16 / 0.21]), Sigma_6040, ["equities", "bonds"])
print(t4["share of risk"].round(4).to_dict())

# 5.
b = np.array([0.3, 0.4, 0.3])
a = w - b
te, mr_te, rc_te = vol_contributions(a, Sigma)
pd.DataFrame({"active weight": a, "marginal TE": mr_te, "TE contribution": rc_te, "share": rc_te / te},
             index=assets).round(4), f"TE = {te:.2%}"

**1.** $\Sigma w = (0.5 \cdot 0.0256 + 0.5 \cdot 0.0008,\ 0.5 \cdot 0.0008 + 0.5 \cdot 0.0025) = (0.0132,\ 0.00165)$,
$\sigma^2 = 0.5 \cdot 0.0132 + 0.5 \cdot 0.00165 = 0.007425$, $\sigma = 8.62\%$. Contributions $0.0066/0.0862 = 7.66\%$ (equities, 88.9%)
and $0.000825/0.0862 = 0.96\%$ (bonds, 11.1%).

**2.** The sum always equals $\sigma$ (Euler). A contribution equals $w_i\sigma_i\rho_{i,p}$, so it is at most $|w_i|\sigma_i$
in absolute value; it reaches it only if the asset is perfectly correlated (or anti-correlated) with the portfolio.

**3.** "Almost all of the portfolio's volatility comes from equities; to change the risk level,
change the equity allocation. The bond allocation can be adjusted for other reasons (liquidity,
liability matching) with little effect on volatility." It does **not** mean bonds are riskless: on
their own they have 5% volatility, and in a rising-rate scenario where bonds and equities fall together
their contribution would be much larger.

**4.** With two assets, $\mathrm{RC}_1 = w_1(w_1\sigma_1^2 + w_2\rho\sigma_1\sigma_2)/\sigma$, $\mathrm{RC}_2 = w_2(w_2\sigma_2^2 + w_1\rho\sigma_1\sigma_2)/\sigma$. The cross terms
are equal, so $\mathrm{RC}_1 = \mathrm{RC}_2 \iff w_1^2\sigma_1^2 = w_2^2\sigma_2^2 \iff w_1\sigma_1 = w_2\sigma_2$ (long-only). With $w_1 + w_2 = 1$:
$w_E = \sigma_B/(\sigma_E + \sigma_B) = 5/21 = 23.8\%$.

**5.** Active weights $+10\%$ Swiss equities, $-10\%$ global equities, 0 bonds; TE 1.05% (04/01, Example 3).
The global-equity underweight contributes 65%, the Swiss-equity overweight 35%, bonds 0 (zero active
weight, whatever their marginal TE).

The contribution is $a_i \cdot \partial\mathrm{TE}/\partial a_i$ with $\partial\mathrm{TE}/\partial a_i = \operatorname{Cov}(R_i, R_A)/\mathrm{TE}$ and active return
$R_A = 0.1\,(R_{SE} - R_{GE})$. For global equities $\operatorname{Cov}(R_{GE}, R_A) = 0.1\,(0.02176 - 0.0289) < 0$: holding **more**
global equities would reduce TE. A negative active weight times a negative marginal TE is a positive
contribution. The underweight dominates because global equities are the more volatile of the two
(17% vs 16%): $|\operatorname{Cov}(R_{GE}, R_A)| = 0.00071$ against $\operatorname{Cov}(R_{SE}, R_A) = 0.00038$.

## 9. Conclusion

- A positively homogeneous risk measure satisfies Euler's theorem: $\rho(w) = \sum_i w_i\,\partial\rho/\partial w_i$.
- Volatility contributions: $\mathrm{RC}_i = w_i(\Sigma w)_i/\sigma = w_i\sigma_i\rho_{i,p}$; they add up to $\sigma$, are at most the stand-alone risk, and are negative for hedges.
- Normal VaR has the same percentage contributions as volatility; ES contributions are the average position loss on the portfolio's worst days, and they reveal jump risk that volatility misses.
- At the tangency portfolio, excess return per unit of marginal risk is the same for every asset; where it differs, it shows the direction to tilt.
- Capital weights are not risk weights: 60/40 is about 94/6 in risk.

Next: [Module 06, 01 How noisy is a mean return?](../06_estimation_error/01_how_noisy_is_a_mean_return.ipynb).